# Phutball Transformer vs CNN Profiler

Runtime → A100 GPU or TPU v6e → Run all

In [ ]:
# Install dependencies
import os
IN_COLAB = 'google.colab' in str(globals()) or os.path.exists('/content')

if IN_COLAB:
    try:
        import jax
        if 'TPU' in str(jax.devices()):
            !pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
        else:
            raise Exception('No TPU')
    except:
        !pip install -q jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    !pip install -q flax optax mctx

In [ ]:
# Clone repo
REPO_URL = "https://github.com/echoname6/phutball-jax.git"
REPO_DIR = "/content/phutball-jax" if IN_COLAB else "./phutball-jax"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# Verify devices
import jax
import jax.numpy as jnp
import time

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Device count: {jax.device_count()}")

DEVICE_TYPE = str(jax.devices()[0]).split(':')[0] if jax.devices() else 'cpu'
print(f"Using {jax.device_count()}x {DEVICE_TYPE}")

In [ ]:
# Imports
from phutball_env_jax import EnvConfig
from network import (
    create_network, init_network,
    create_transformer_network, init_transformer_network,
)
from self_play_batched import (
    play_games_batched,
    make_mcts_recurrent_fn,
    batched_mcts_policy,
    batched_reset,
    make_batched_step,
    make_batched_legal_actions,
    make_batched_network_input,
    transformer_mcts_policy,
    make_transformer_recurrent_fn,
)

print("Imports OK")

In [ ]:
# === CONFIG ===
ROWS, COLS = 15, 11

# CNN config
CNN_CHANNELS = 128
CNN_BLOCKS = 10

# Transformer config
D_MODEL = 128
N_LAYERS = 6
N_HEADS = 4
FFN_DIM = 256

env_config = EnvConfig(rows=ROWS, cols=COLS)
print(f"Board: {ROWS}x{COLS}, action space: {2*ROWS*COLS+1}")

In [ ]:
# Initialize both networks
rng = jax.random.PRNGKey(42)

# CNN
rng, init_rng = jax.random.split(rng)
cnn_network = create_network(rows=ROWS, cols=COLS, num_channels=CNN_CHANNELS, num_res_blocks=CNN_BLOCKS)
cnn_variables = init_network(init_rng, cnn_network, num_input_channels=6)
cnn_params = {
    'network_params': cnn_variables['params'],
    'batch_stats': cnn_variables['batch_stats'],
}
cnn_param_count = sum(x.size for x in jax.tree_util.tree_leaves(cnn_variables['params']))

# Transformer
rng, init_rng = jax.random.split(rng)
tf_network = create_transformer_network(
    rows=ROWS, cols=COLS, d_model=D_MODEL,
    n_layers=N_LAYERS, n_heads=N_HEADS, ffn_dim=FFN_DIM,
)
tf_variables = init_transformer_network(init_rng, tf_network, num_input_channels=6)
tf_params = {'network_params': tf_variables['params']}
tf_param_count = sum(x.size for x in jax.tree_util.tree_leaves(tf_variables['params']))

print(f"CNN:         {CNN_CHANNELS}ch, {CNN_BLOCKS} blocks, {cnn_param_count:,} params")
print(f"Transformer: d={D_MODEL}, {N_LAYERS}L, {N_HEADS}H, ffn={FFN_DIM}, {tf_param_count:,} params")

## 1. Network Throughput

In [ ]:
def bench_throughput(network, variables, label, batch_sizes, num_iters=50):
    """Benchmark raw forward pass throughput."""
    rng = jax.random.PRNGKey(0)

    @jax.jit
    def forward(x):
        return network.apply(variables, x, train=False)

    print(f"\n{label}:")
    results = {}
    for batch in batch_sizes:
        x = jax.random.normal(rng, (batch, 6, ROWS, COLS))
        p, v = forward(x)
        p.block_until_ready()

        start = time.perf_counter()
        for _ in range(num_iters):
            p, v = forward(x)
        p.block_until_ready()
        elapsed = time.perf_counter() - start

        sps = (batch * num_iters) / elapsed
        ms = (elapsed / num_iters) * 1000
        results[batch] = sps
        print(f"  batch={batch:4d}: {sps:>10,.0f} samples/sec  ({ms:.1f}ms/batch)")
    return results

batch_sizes = [32, 64, 128, 256, 512]

cnn_tp = bench_throughput(cnn_network, cnn_variables, "CNN", batch_sizes)
tf_tp = bench_throughput(tf_network, {'params': tf_variables['params']}, "Transformer", batch_sizes)

print("\nSpeedup (CNN / Transformer):")
for b in batch_sizes:
    if b in cnn_tp and b in tf_tp:
        print(f"  batch={b:4d}: {cnn_tp[b]/tf_tp[b]:.2f}x")

## 2. MCTS Policy

In [ ]:
def bench_mcts(network, params, policy_fn, recurrent_fn, label,
               batch_sizes, num_simulations=32, num_calls=10):
    """Benchmark MCTS policy calls."""
    rng = jax.random.PRNGKey(0)
    print(f"\n{label} (sims={num_simulations}):")
    results = {}

    for batch_size in batch_sizes:
        try:
            states = batched_reset(env_config, batch_size)

            # Warmup
            rng, prng = jax.random.split(rng)
            actions, _, _ = policy_fn(
                params, states, prng, network, env_config,
                num_simulations=num_simulations, temperature=1.0,
                recurrent_fn=recurrent_fn,
            )
            actions.block_until_ready()

            start = time.perf_counter()
            for _ in range(num_calls):
                rng, prng = jax.random.split(rng)
                actions, _, _ = policy_fn(
                    params, states, prng, network, env_config,
                    num_simulations=num_simulations, temperature=1.0,
                    recurrent_fn=recurrent_fn,
                )
            actions.block_until_ready()
            elapsed = time.perf_counter() - start

            total_sims = batch_size * num_simulations * num_calls
            sps = total_sims / elapsed
            ms = (elapsed / num_calls) * 1000
            results[batch_size] = sps
            print(f"  batch={batch_size:4d}: {sps:>10,.0f} sims/sec  ({ms:.0f}ms/call)")
        except Exception as e:
            print(f"  batch={batch_size:4d}: FAILED - {e}")
    return results

mcts_batches = [32, 64, 128, 256]
NUM_SIMS = 32

cnn_recurrent = make_mcts_recurrent_fn(cnn_network, env_config)
tf_recurrent = make_transformer_recurrent_fn(tf_network, env_config)

cnn_mcts = bench_mcts(cnn_network, cnn_params, batched_mcts_policy, cnn_recurrent,
                       "CNN", mcts_batches, NUM_SIMS)
tf_mcts = bench_mcts(tf_network, tf_params, transformer_mcts_policy, tf_recurrent,
                      "Transformer", mcts_batches, NUM_SIMS)

print("\nSpeedup (CNN / Transformer):")
for b in mcts_batches:
    if b in cnn_mcts and b in tf_mcts:
        print(f"  batch={b:4d}: {cnn_mcts[b]/tf_mcts[b]:.2f}x")

## 3. Full Self-Play Games

In [ ]:
def bench_full_games(network, params, label,
                     batch_size=64, num_simulations=32, max_turns=100,
                     mcts_policy_fn=None, recurrent_fn=None):
    """Benchmark full game self-play."""
    rng = jax.random.PRNGKey(42)

    # Warmup
    print(f"\n{label}: warming up...")
    rng, grng = jax.random.split(rng)
    traj = play_games_batched(
        params=params, rng=grng, network=network, env_config=env_config,
        batch_size=min(batch_size, 16), max_turns=10, max_moves=50,
        temperature=1.0, num_simulations=min(num_simulations, 16),
        mcts_policy_fn=mcts_policy_fn, recurrent_fn=recurrent_fn,
    )
    _ = traj.winners.block_until_ready()

    # Benchmark
    print(f"  Playing {batch_size} games (sims={num_simulations}, max_turns={max_turns})...")
    rng, grng = jax.random.split(rng)
    start = time.perf_counter()
    traj = play_games_batched(
        params=params, rng=grng, network=network, env_config=env_config,
        batch_size=batch_size, max_turns=max_turns, max_moves=max_turns * 3,
        temperature=1.0, temp_threshold=30, temp_final=0.1,
        num_simulations=num_simulations,
        mcts_policy_fn=mcts_policy_fn, recurrent_fn=recurrent_fn,
    )
    _ = traj.winners.block_until_ready()
    elapsed = time.perf_counter() - start

    total_moves = int(traj.valid_mask.sum())
    total_sims = total_moves * num_simulations
    winners = traj.winners
    p1 = int((winners == 1).sum())
    p2 = int((winners == 2).sum())
    draws = int((winners == 0).sum())

    print(f"  Time:       {elapsed:.1f}s")
    print(f"  Moves:      {total_moves} ({total_moves/batch_size:.1f}/game)")
    print(f"  Games/sec:  {batch_size/elapsed:.3f}")
    print(f"  Moves/sec:  {total_moves/elapsed:.1f}")
    print(f"  Sims/sec:   {total_sims/elapsed:,.0f}")
    print(f"  W1/W2/D:    {p1}/{p2}/{draws}")
    return total_sims / elapsed

GAME_BATCH = 64
GAME_SIMS = 32
GAME_TURNS = 100

cnn_sps = bench_full_games(
    cnn_network, cnn_params, "CNN",
    batch_size=GAME_BATCH, num_simulations=GAME_SIMS, max_turns=GAME_TURNS,
)
tf_sps = bench_full_games(
    tf_network, tf_params, "Transformer",
    batch_size=GAME_BATCH, num_simulations=GAME_SIMS, max_turns=GAME_TURNS,
    mcts_policy_fn=transformer_mcts_policy, recurrent_fn=tf_recurrent,
)

print(f"\nFull games speedup (CNN / Transformer): {cnn_sps/tf_sps:.2f}x")

## 4. Game Logic vs NN Breakdown

In [ ]:
def bench_breakdown(network, params, label, use_transformer=False, batch_size=256):
    """Time game logic components vs NN forward pass."""
    states = batched_reset(env_config, batch_size)
    actions = jnp.zeros(batch_size, dtype=jnp.int32)

    step_fn = make_batched_step(env_config)
    legal_fn = make_batched_legal_actions(env_config)
    input_fn = make_batched_network_input(env_config)

    if use_transformer:
        variables = {'params': params['network_params']}
    else:
        variables = {'params': params['network_params'], 'batch_stats': params['batch_stats']}

    @jax.jit
    def nn_forward(x):
        return network.apply(variables, x, train=False)

    # Warmup
    _ = step_fn(states, actions).board.block_until_ready()
    _ = legal_fn(states).block_until_ready()
    inputs = input_fn(states)
    _ = nn_forward(inputs)[0].block_until_ready()

    N = 100
    timings = {}

    t0 = time.perf_counter()
    for _ in range(N):
        r = step_fn(states, actions)
    r.board.block_until_ready()
    timings['step'] = (time.perf_counter() - t0) / N

    t0 = time.perf_counter()
    for _ in range(N):
        r = legal_fn(states)
    r.block_until_ready()
    timings['legal'] = (time.perf_counter() - t0) / N

    t0 = time.perf_counter()
    for _ in range(N):
        inputs = input_fn(states)
    inputs.block_until_ready()
    timings['convert'] = (time.perf_counter() - t0) / N

    t0 = time.perf_counter()
    for _ in range(N):
        p, v = nn_forward(inputs)
    p.block_until_ready()
    timings['nn'] = (time.perf_counter() - t0) / N

    total = sum(timings.values())
    print(f"\n{label} (batch={batch_size}):")
    for k, v in timings.items():
        print(f"  {k:>12s}: {v*1000:6.2f}ms ({100*v/total:5.1f}%)")
    print(f"  {'TOTAL':>12s}: {total*1000:6.2f}ms")
    return timings

cnn_bd = bench_breakdown(cnn_network, cnn_params, "CNN", use_transformer=False)
tf_bd = bench_breakdown(tf_network, tf_params, "Transformer", use_transformer=True)

print(f"\nNN forward: CNN={cnn_bd['nn']*1000:.2f}ms, Transformer={tf_bd['nn']*1000:.2f}ms")
print(f"  Ratio: {cnn_bd['nn']/tf_bd['nn']:.2f}x")

## 5. Summary

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"\nDevice: {jax.device_count()}x {DEVICE_TYPE}")
print(f"Board:  {ROWS}x{COLS}")
print(f"\nCNN:         {CNN_CHANNELS}ch, {CNN_BLOCKS} blocks, {cnn_param_count:,} params")
print(f"Transformer: d={D_MODEL}, {N_LAYERS}L, {N_HEADS}H, ffn={FFN_DIM}, {tf_param_count:,} params")
print(f"\nFull-game sims/sec:")
print(f"  CNN:         {cnn_sps:>10,.0f}")
print(f"  Transformer: {tf_sps:>10,.0f}")
print(f"  Ratio:       {cnn_sps/tf_sps:.2f}x")
print(f"\nNN forward (batch=256):")
print(f"  CNN:         {cnn_bd['nn']*1000:.2f}ms")
print(f"  Transformer: {tf_bd['nn']*1000:.2f}ms")
print(f"  Ratio:       {cnn_bd['nn']/tf_bd['nn']:.2f}x")

# Training time estimate
train_batch = 256
train_sims = 32
moves_per_game = 170
total_sims = train_batch * moves_per_game * train_sims
print(f"\nEstimated self-play time per iteration ({train_batch} games, {train_sims} sims):")
print(f"  CNN:         {total_sims/cnn_sps:.0f}s ({total_sims/cnn_sps/60:.1f}min)")
print(f"  Transformer: {total_sims/tf_sps:.0f}s ({total_sims/tf_sps/60:.1f}min)")